# OPFModel Paper-to-KG Ingestion Notebook

This notebook handles offline paper ingestion only:

- Read classes, properties, and taxonomy whitelists from `ontology/OPF_Ontology2.0.owl`.
- Extract PaperModel / OPFModel information from local OPF paper files in PDF, TXT, or MD format.
- After manual preview and high-confidence review, write the paper's OPFModel individual to the integrated knowledge graph at `data/01_paper_ingestion_workflow/03_output/OPF_Knowledge_Graph2.0.owl`.
- Support both single-paper ingestion and batch scanning of `data/01_paper_ingestion_workflow/01_input/01_source_papers`. The public repository does not include third-party PDFs in this directory by default; users must obtain them lawfully and place local copies there.
- The authoritative public bibliography for the 36 papers currently represented in the knowledge graph is stored in `data/01_paper_ingestion_workflow/01_input/01_source_papers/paper_corpus_manifest.csv`; workflow 02 retains a byte-identical reference mirror. P036 is the newly added DeepOPF-V paper and does not yet have a corresponding paper-motivation requirement; the fixed paper-ingestion validation benchmark remains at 35 papers. Online recommendation and the existing validation workflow read the knowledge graph directly rather than the source PDFs.

File roles:

- `OPF_Ontology2.0.owl`: the general ontology schema.
- `OPF_Knowledge_Graph2.0.owl`: the integrated knowledge graph containing the ontology schema and individual paper-model instances.

This notebook is not responsible for maintaining the underlying ontology schema or providing online user recommendations.


## 1. Environment and Path Setup

**This section is the environment entry point for the paper-ingestion workflow.**

The cell imports the packaged project library and shared paths. Subsequent PDF reading, LLM extraction, intermediate-file storage, and OWL writing all reuse this configuration.

Note: the core ontology taxonomy/schema is fixed in `ontology/OPF_Ontology2.0.owl` and `data/01_paper_ingestion_workflow/03_output/OPF_Knowledge_Graph2.0.owl`; this notebook is responsible only for writing PaperModel instances derived from papers.


In [ ]:
# Project package bootstrap.
# Reusable functions are imported from src/opfmodelselection instead of being defined in notebook cells.

import sys
from pathlib import Path

_CWD = Path.cwd().resolve()
for _candidate in [_CWD, _CWD.parent]:
    if (_candidate / "src" / "opfmodelselection").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise RuntimeError("Cannot find src/opfmodelselection from current working directory.")

from src.opfmodelselection.config import *
print("项目配置已加载 / Project config loaded:", PROJECT_ROOT)


## 1.1 Shared LLM/API Configuration

This section loads the centralized LLM/API configuration for the paper-ingestion workflow.

Confirm the following settings here:

- Whether the OpenAI / RWTH API key is available.
- Which LLM route is currently selected.
- Which model name and temperature are in use.

To change the model name, route selection, temperature, or API client, edit only `src/opfmodelselection/config.py`. The notebook consumes this configuration without redefining it.


In [ ]:
# Shared LLM/API configuration is imported from src.opfmodelselection.config.
print_llm_config_status()


## 2. Load the OPF Ontology and Knowledge Graph

Check and read the following two OWL files:

- `OPF_Ontology2.0.owl`: the ontology schema containing classes, properties, and the taxonomy.
- `OPF_Knowledge_Graph2.0.owl`: the integrated knowledge graph containing the ontology schema and paper-derived OPFModel individuals.

This section checks only the paths and XML parseability; it neither rebuilds the schema nor writes to the knowledge graph.


In [ ]:
# Check ontology paths and read the OPF ontology XML.
check_project_paths(include_oeo_model=True)
setup_owlready_paths(include_oeo_model=True)
OPF_XML_TEXT, OPF_XML_ROOT, OPF_OWL_IMPORTS = read_opf_xml()
print_opf_xml_status()


## 8. Offline Paper-to-OPFModel KG Ingestion

Input: one local OPF paper in PDF, TXT, or MD format.

Output: a previewed and reviewed `opf:OPFModel` individual that is ultimately written to `OPF_Knowledge_Graph2.0.owl`.

Recommended sequence:

1. **8.1-8.2** Read the paper text, tables, and images.
2. **8.3** Read the taxonomy/object-property whitelists from the schema ontology.
3. **8.4-8.5** Call the LLM to generate `paper_model_record`.
4. **8.7** Preview the record before writing.
5. **8.8** Review and prune high-confidence errors.
6. **8.9** Write to `OPF_Knowledge_Graph2.0.owl`.
7. **8.9.1** Use the current PDF to look up the newly written instance.

Only Sections 8.9 and 8.10 modify the knowledge graph; Sections 8.1-8.8 do not write OWL.


### 8.1 PDF, Image, and Table Reader Imports

**This subsection only imports paper-reading functions; it neither calls the LLM nor writes OWL.**

It converts a local paper in PDF, TXT, or MD format into `paper_record` for the subsequent LLM extraction step.

Main outputs:

- `paper_record["text"]`: the nearly complete paper text.
- `paper_record["tables"]`: structured tables extracted from the PDF.
- `paper_record["images"]`: information about images extracted from the PDF or rendered page by page.
- `paper_record["llm_payload"]["prompt_text"]`: the text input subsequently sent to the LLM.
- `paper_record["llm_payload"]["image_input_paths"]`: paths of images subsequently sent to the multimodal model.

This subsection first decomposes the paper into text, tables, and images so that the LLM can extract the OPFModel from information covering as much of the full paper as possible rather than from the abstract alone.


In [ ]:
# PDF, image, table and optional OCR processing functions.
from src.opfmodelselection.pdf_processing import *
print("PDF 处理封装已加载 / PDF processing wrapper loaded.")
print("当前论文路径 / Configured PAPER_PATH:", PAPER_PATH)


### 8.2 Run Paper Processing Locally

**This subsection checks the quality of paper reading; it neither calls the LLM nor writes OWL.**

Set `PAPER_PATH` here, then run the code to generate `paper_record` and `paper_record.json`.

What to check:

- `Extracted text length`: the extracted body-text length; a very short value may indicate a scanned PDF or failed extraction.
- `OCR fallback used`: whether the OCR fallback was enabled automatically.
- `Images passed to multimodal LLM`: how many images will be sent to the multimodal model.
- `LLM prompt preview`: whether the beginning of the text sent to the LLM resembles the paper body rather than corrupted text.

If extraction quality is poor in this subsection, do not continue to Section 8.5. Fix PDF reading first; otherwise, the subsequently extracted OPFModel information may be inaccurate.


In [ ]:
# Install dependencies if they are not already installed.
# Install once from the project root: .\.venv312\Scripts\python.exe -m pip install -r requirements.txt

print_paper_processing_status()

# Example: replace this with the actual path of your local paper.
# PAPER_PATH = PAPER_DIR / "your_paper.pdf"
if PAPER_PATH is not None:
    print("正在处理论文 / Processing:", PAPER_PATH)
    paper_record = extract_paper_record(PAPER_PATH)
    record_path = save_paper_record(paper_record)
    print("已保存 paper_record / Saved paper record:", record_path)
    print("提取文本长度 / Extracted text length:", len(paper_record["text"]))
    print("是否启用 OCR fallback / OCR fallback used:", paper_record["ocr_fallback_used"])
    print("提取到的内嵌图片数量 / Embedded images extracted:", paper_record["embedded_image_count"])
    print("渲染出的整页图片数量 / Rendered page images:", paper_record["page_render_image_count"])
    print("传给多模态 LLM 的图片路径 / Images passed to multimodal LLM:", paper_record["llm_payload"]["image_input_paths"])
    print("LLM 文本 prompt 预览 / LLM prompt preview:\n", paper_record["llm_payload"]["prompt_text"][:2000])
else:
    print("请先设置 PAPER_PATH 为本地 PDF/TXT/MD 文件，然后重新运行本单元格 / Set PAPER_PATH and rerun this cell.")


### 8.3 Ontology Whitelist Retrieval

**This subsection reads classification whitelists and parent-child class relationships from `ontology/OPF_Ontology2.0.owl`.**

Rather than hard-coding whitelists, it directly queries the taxonomy defined in the ontology. When the hierarchy in the OWL file changes, rerunning this section retrieves the latest class names.

This subsection generates:

- `OBJECTIVE_LEVEL1_CLASSES`, `OBJECTIVE_LEVEL2_CLASSES`, `OBJECTIVE_LEVEL2_BY_LEVEL1`
- `CONSTRAINT_LEVEL1_CLASSES`, `CONSTRAINT_LEVEL2_CLASSES`, `CONSTRAINT_LEVEL2_BY_LEVEL1`
- `PARAMETER_LEVEL1_CLASSES`, `PARAMETER_LEVEL2_CLASSES`, `PARAMETER_LEVEL2_BY_LEVEL1`
- `FEATURE_ALLOWED_CLASSES_BY_PROPERTY`: knowledge-graph classes that are still written as object properties, currently `hasMathematicalProblemType`, `hasModeledTechnology`, and OEO `covers sector`.

Important rules:

- During paper extraction, the LLM primarily selects second-level Objective, Constraint, and Parameter classes.
- Python can infer first-level classes automatically from the OWL parent-child relationships.
- Before writing, the workflow still verifies that each second-level class genuinely belongs to its corresponding first-level class.

This subsection neither calls the LLM nor writes OWL. It only prepares the knowledge-graph whitelists required for object-property class selection in the second pass; model features that are now represented as data-property text are no longer included in the enum whitelist.


In [ ]:
# Ontology whitelist retrieval functions and current whitelist tables.
from src.opfmodelselection.ontology_whitelist import *
print("Ontology whitelist loaded from schema OWL file:", BASE_OWL_PATH)
print("目标函数二级类数量 / Objective level-2 choices:", len(OBJECTIVE_LEVEL2_CLASSES))
print("约束二级类数量 / Constraint level-2 choices:", len(CONSTRAINT_LEVEL2_CLASSES))
print("参数变量二级类数量 / Parameter level-2 choices:", len(PARAMETER_LEVEL2_CLASSES))
for prop, choices in FEATURE_ALLOWED_CLASSES_BY_PROPERTY.items():
    print(f"- 对象特征属性 / object feature property {prop}: {len(choices)} 个候选 / choices")


### 8.4 Two-Pass LLM Extraction

**This subsection defines the two-pass LLM extraction functions; the API is actually called in Section 8.5.**

The workflow uses two passes because sending every whitelist to the LLM at once creates an excessively large input and encourages incorrect selections. The task is therefore split into two clearer structured-extraction steps.

The two passes:

1. **Data-property pass**: extract OPFModel data properties, including paper metadata, model purpose, primary output, model-feature text, and textual summaries of objectives, constraints, and parameters.
2. **Object-property pass**: select only classes that are still written as object properties from the knowledge-graph whitelists: second-level Objective, Constraint, and Parameter classes, plus MathematicalProblemType, OEO energy technology, and OEO energy demand sector. Only classes meeting the confidence threshold are retained.

Key constraints:

- `hasModelName` is derived from keywords in the paper title and does not require an OPF suffix.
- `hasPrimaryOutput` must describe the output of running the OPF model, not a paper contribution or algorithmic artifact.
- An Objective can be selected only when it genuinely appears as a `min` or `max` objective in the paper's equations.
- The second pass returns `none` or an empty list when no suitable class exists.
- Second-pass class selections must include confidence values; by default, only results with confidence `>= 0.70` are retained.
- The OpenAI route uses the hard constraint `response_format/json_schema`, while the RWTH route retains the soft-constraint workflow.


In [ ]:
# PaperModel LLM extraction functions.
from src.opfmodelselection.paper_extraction import *
print("两轮单篇论文抽取函数已加载 / Two-pass single-paper extraction functions loaded.")
print("运行方式 / Run: paper_model_record = extract_paper_model_record(PAPER_PATH)")


### 8.5 Run Two-Pass Paper Extraction

**This subsection actually calls the LLM and generates `paper_model_record`.**

Before running, confirm that:

- Section 8.2 has set `PAPER_PATH`.
- Section 8.3 has read the latest whitelists from OWL.
- Section 8.4 has loaded the two-pass LLM extraction functions.

Key intermediate results after execution:

- `paper_model_record["metadata"]`: data properties extracted in the first pass.
- `paper_model_record["structure"]`: objective, constraint, and parameter classifications extracted in the second pass.
- `paper_model_record["features"]`: the small set of object-property model-feature classifications extracted in the second pass.
- `paper_model_record["paper_record"]`: the original paper-extraction record, including body text, tables, and image paths.
- `paper_model_record["paper_model_record_path"]`: the path of the intermediate JSON file saved locally.

This subsection still does not write OWL. It only transforms the paper into a structured record awaiting inspection, review, and writing.


In [ ]:
# Run the actual two-pass LLM extraction for the current paper.

if PAPER_PATH is None:
    raise RuntimeError("Set PAPER_PATH to a local paper PDF/TXT/MD before running extraction.")

paper_model_record = extract_paper_model_record(PAPER_PATH)

print("paper_model_record 已创建 / paper_model_record created.")
print("中间 JSON 文件 / Intermediate JSON:", paper_model_record.get("paper_model_record_path"))
print("metadata 字段 / Metadata fields:", list(paper_model_record["metadata"].keys()))
print("结构类选择结果 / Structure selections:")
print(json.dumps(paper_model_record["structure"], ensure_ascii=False, indent=2))
print("模型特征选择结果 / Feature selections:")
print(json.dumps(paper_model_record["features"], ensure_ascii=False, indent=2))


### 8.6 OPFModel Instance Writer Functions

**This subsection only defines functions; it neither creates instances nor writes OWL.**

It converts the `paper_model_record` produced in Section 8.5 into an `opf:OPFModel` individual declaration that can be written to OWL, and provides all validation and replacement logic required before the final write.

Key capabilities defined in this subsection:

- Validate that every class name in `paper_model_record` is included in the OWL whitelists.
- Automatically add missing first-level classes based on second-level classes, for example `GeneratorCostCoefficientParameter -> EconomicMarketEmissionParameter`.
- Generate the XML fragment for an OPFModel individual.
- Print a complete instance report covering metadata, data properties, object properties, and relationships.
- Check that the core knowledge-graph structure remains intact to avoid writing to a corrupted base ontology.
- Decide whether to replace an existing paper instance: match by DOI first, or by stable paper metadata when no DOI is available.
- When replacing an existing instance, print its metadata and attempt to locate the corresponding paper path under `data/01_paper_ingestion_workflow/01_input/01_source_papers/`.

After running this subsection, the next step is normally the preview in Section 8.7 rather than an immediate write.


In [ ]:
# PaperModel ontology writing functions.
from src.opfmodelselection.paper_writer import *
print("单篇论文模型实例写入函数已加载 / Single-paper model instance writing functions loaded.")
print("运行方式 / Run: write_single_paper_model_instance_to_ontology(paper_model_record)")


### 8.7 Preview the OPFModel Individual Before Writing

**This subsection only previews the individual; it does not write OWL.**

It prints the complete OPFModel instance that the current `paper_model_record` will generate so that it can be inspected before the actual write.

Recommended checks:

- Whether the model instance name is reasonable.
- Whether the paper title, authors, institutions, year, and DOI are correct.
- Whether `hasPrimaryOutput` genuinely describes the output of running the OPF model.
- Whether the data-property text for `hasObjectiveFunction`, `hasConstraintFunction`, and `usesParameter` resembles the paper's content.
- Whether too many or obviously incorrect Objective, Constraint, or Parameter object properties were selected.
- Most model features are written as data-property text; only the mathematical problem type and energy sector remain object properties.

If the preview is unsatisfactory, do not run Section 8.9. Rerun Section 8.5, or manually adjust `paper_model_record` and preview it again.


In [ ]:
# Preview only. This cell does not modify OPF_Knowledge_Graph2.0.owl.

if "paper_model_record" not in globals():
    raise RuntimeError("Run section 8.5 first: paper_model_record = extract_paper_model_record(PAPER_PATH).")

preview_report = print_paper_model_instance_report(paper_model_record)


### 8.8 LLM High-Confidence Review Before the Final Write

**This subsection is the second safeguard before writing and still does not write OWL.**

It resubmits the extracted paper text, tables, and image information together with the structured results from the first two LLM passes so that the model can specifically check for clear errors.

The review rules are deliberately conservative:

- Only remove incorrect items or change clearly incorrect data properties to `None`.
- Do not add, replace, or rename classes, and do not invent classes absent from the taxonomy.
- Remove an item only when confidence that it is incorrect is at least 80%.
- Retain uncertain content for human judgment during preview.

The review updates `paper_model_record`. Afterward, rerun Section 8.7 to inspect the final content that will be written.


In [ ]:
# LLM high-confidence review gate. This cell does not modify OPF_Knowledge_Graph2.0.owl.

if "paper_model_record" not in globals():
    raise RuntimeError("请先运行 8.5 生成 paper_model_record / Run section 8.5 first.")

paper_model_record = review_and_prune_paper_model_record(paper_model_record)
print_review_summary(paper_model_record)

print("\n复核后的 OPFModel 实例预览 / Preview after review pruning:")
preview_report = print_paper_model_instance_report(paper_model_record)


### 8.9 Final Write to the OPF Knowledge Graph

**This is the only subsection in Section 8 that directly modifies `OPF_Knowledge_Graph2.0.owl`.**

Run this cell only after confirming that both the Section 8.7 preview and Section 8.8 review are correct.

Write logic:

- The default `append=True` preserves other paper-model instances that were written previously.
- If an existing instance has the same DOI, replace it to prevent one paper from being represented by two OPFModel instances.
- If no DOI is available, use stable paper metadata such as title, author, institution, and year to determine whether the paper is the same.
- If only the LLM-extracted `hasModelName` differs while the DOI or stable metadata matches, replace the existing instance.
- If the paper or model is genuinely different, create a new OPFModel instance.

Write protection:

- Check first that the core knowledge-graph structure exists.
- Automatically add first-level classes inferred from second-level classes.
- Print the final write report.
- When replacing an existing instance, print its metadata and attempt to identify the corresponding paper path under `data/01_paper_ingestion_workflow/01_input/01_source_papers/`.

After successful execution, `OPF_Knowledge_Graph2.0.owl` permanently retains this paper-model instance unless it is later deleted or replaced explicitly.


In [ ]:
# FINAL WRITE CELL. Run only after checking the 8.8 review and preview.
#
# Important: this cell itself is responsible only for calling the write function.
# The actual pre-write validation logic is defined in the previous section:
#   assert_core_knowledge_graph_ready_for_paper_write(xml_text)
# The current version no longer requires the legacy taxonomy marker; it checks actual OWL classes, parent-child relationships, and property definitions.

if "paper_model_record" not in globals():
    raise RuntimeError("请先运行 8.5 生成 paper_model_record，然后运行 8.8 复核和预览 / Run 8.5, then 8.8 before writing.")

print("8.9 写入前检查 / pre-write guard:")
print("- 不再要求旧 taxonomy 自动生成 marker。")
print("- 会检查真实 OWL 类、父子类关系、PaperModel/OPFModel 属性定义。")
print("- 如果检查通过，才会写入当前论文 OPFModel 实例。")

paper_model_record = write_single_paper_model_instance_to_ontology(paper_model_record)


### 8.9.1 Verify the Written OPFModel Using the Current PDF

**This subsection does not modify OWL; it only reads and prints data.**

Usage scenario:

1. Section 8.9 has written the paper identified by the current `PAPER_PATH` to `OPF_Knowledge_Graph2.0.owl`.
2. Use the PDF's paper title and DOI to retrieve the corresponding `opf:OPFModel` instance from the knowledge graph.
3. After a match is found, print all of its data and object properties in the PaperModel ingestion format and translate the data-property values themselves into Chinese.

Matching rules:

- Prefer exact DOI matching.
- If the DOI is missing or not found, match using the normalized paper title.
- Support both legacy relative IRIs and current full IRIs.


In [ ]:
# 8.9.1 Read back the written OPFModel individual for the current PAPER_PATH.
from src.opfmodelselection.paper_lookup import *

current_pdf_title, current_pdf_doi = extract_title_doi_from_current_pdf()
matched_opfmodel_record, match_reason = find_opfmodel_by_pdf_metadata(current_pdf_title, current_pdf_doi)

if matched_opfmodel_record is None:
    print("未找到匹配当前 PDF title/DOI 的 OPFModel 实例 / No OPFModel individual matched the current PDF title/DOI.")
    print("PDF 论文标题 / PDF paper title:", current_pdf_title)
    print("PDF DOI:", current_pdf_doi)
else:
    current_pdf_opfmodel_record = print_opfmodel_in_papermodel_format(
        matched_opfmodel_record,
        match_reason=match_reason,
        lookup_title=current_pdf_title,
        lookup_doi=current_pdf_doi,
    )


### 8.10 Standalone Batch Ingestion from the Source-Paper Folder

**This subsection directly modifies `OPF_Knowledge_Graph2.0.owl`.**

Section 8.10 is designed as a standalone batch entry point:

- First run only **Section 1 for path/environment configuration** and **Section 2 for ontology loading**.
- Then run Section 8.10 directly.
- Section 8.10 automatically loads the required Section 8 function definitions and whitelists:
  - Section 8.1 PDF, image, and table extraction wrappers
  - Section 8.3 objective, constraint, parameter, and feature whitelists dynamically read from OWL
  - Section 8.4 two-pass LLM extraction functions
  - Section 8.6 OPFModel instance-writing functions
- Section 8.10 does not automatically run the single-paper execution cells:
  - Section 8.5 single-paper extraction execution
  - Section 8.7 preview
  - Section 8.8 single-paper review execution
  - Section 8.9 single-paper final write

Batch workflow:

1. Automatically scan PDFs under `data/01_paper_ingestion_workflow/01_input/01_source_papers/`.
2. First extract DOI and title candidates from each PDF without using an LLM.
3. Check whether OWL already contains the corresponding `opf:OPFModel` instance, preferring DOI and then title.
4. If the instance already exists, skip it without calling the LLM.
5. If it does not exist, run the complete offline workflow: PDF extraction → two-pass LLM extraction → high-confidence review → OWL writing.
6. Continue automatically to the next paper after processing each one.

The main value to adjust is:

```python
BATCH_NEW_PAPER_WRITE_LIMIT = 2
```

This value is the maximum number of previously absent paper instances to add during the current run. Existing papers are skipped and do not consume this insertion limit.


In [ ]:
# 8.10 Batch ingestion can run after Sections 1 and 2 because all functions are packaged.
from src.opfmodelselection.batch_ingestion import *

batch_ingestion_summary = ingest_new_papers_from_folder(BATCH_NEW_PAPER_WRITE_LIMIT)


## 8.11+ Validation Moved

The current validation and evaluation workflow is located in `src/Validation2.0.ipynb`.

This keeps the current notebook limited to offline paper extraction and OPFModel instance writing.
